In [1]:
# !pip install -qU langchain langchain_openai langgraph arxiv duckduckgo-search
# !pip install -qU faiss-cpu pymupdf

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import ArxivLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

/Users/jeevan-11987/Personal/ai-resources-guide/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/jeevan-11987/Personal/ai-resources-guide/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1005 21:57:28.335000 7984 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/var/folders/dc/kw3ll4h16l593j6myd2kj7vr0000gp/T/ipykernel_7984/2265781910.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone int

In [16]:
# Load the document pertaining to a particular topic
docs = ArxivLoader(query="2103.07676", top_k_results=1).load()
print("Fetched following documents:")
for i in range(len(docs)):
    doc = docs[i]
    print(f"{i + 1}. {doc.metadata['Title']}")

# Split the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=350, chunk_overlap=50
)
chunked_documents = text_splitter.split_documents(docs)
# Instantiate the Embedding Model
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Create Index- Load document chunks into the vectorstore
faiss_vectorstore = FAISS.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
)
# Create a retriver
retriever = faiss_vectorstore.as_retriever()

Fetched following documents:
1. uTHCD: A New Benchmarking for Tamil Handwritten OCR


In [17]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12f0259d0>, search_kwargs={})

In [18]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Use the following context to answer the user's query. If you cannot answer the question, please respond with 'I don't know'.

Question:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

In [19]:
from langchain_ollama import ChatOllama

In [20]:
model = ChatOllama(
    model="llama3.2:latest",
    validate_model_on_init=True,
    temperature=0.8,
    num_predict=256,
)

In [21]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

retrieval_augmented_generation_chain = (
       {"context": itemgetter("question") 
    | retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | model, "context": itemgetter("context")}
)

In [22]:
d = {"key1": "val1", "key2": "val2"}
fn = itemgetter("key2")
print(fn(d))
print(type(fn))

val2
<class 'operator.itemgetter'>


In [23]:
retrieval_augmented_generation_chain

{
  context: RunnableLambda(itemgetter('question'))
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12f0259d0>, search_kwargs={}),
  question: RunnableLambda(itemgetter('question'))
}
| RunnableAssign(mapper={
    context: RunnableLambda(itemgetter('context'))
  })
| {
    response: ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the user's query. If you cannot answer the question, please respond with 'I don't know'.\n\nQuestion:\n{question}\n\nContext:\n{context}\n"), additional_kwargs={})])
              | ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='llama3.2:latest', validate_model_on_init=True, num_predict=256, t

In [24]:
await retrieval_augmented_generation_chain.ainvoke({"question" : "What is uTHCD"})

{'response': AIMessage(content='uTHCD stands for Unconstrained Tamil Handwritten Character Database. It is a large database of handwritten characters in the Tamil script, consisting of around 91000 samples with nearly 600 samples in each of 156 classes. The database is a unified collection of both online and offline samples and is expected to set a new benchmark in Tamil handwritten character recognition.', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-10-05T16:45:59.625104Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6765849042, 'load_duration': 60767875, 'prompt_eval_count': 2975, 'prompt_eval_duration': 5231256834, 'eval_count': 73, 'eval_duration': 1436094042, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--01a10cf5-1d58-7ec0-8ed1-df3a83e223e6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2975, 'output_tokens': 73, 'total_tokens': 3048}),
 'context': [Docu

In [25]:
for doc in docs:
    print(doc.metadata["Title"])

uTHCD: A New Benchmarking for Tamil Handwritten OCR
